# Studio 2 - Klasifikasi Sampah (Kelompok 8)
Model 1: ResNet-18 pretrained. Hasil training disimpan ke `submission/model.pth`.

In [ ]:
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from evaluate import metrics
from submission.model import get_model, get_transform

DATA = "./train"
EPOCHS = 10
LR = 3e-4
BATCH = 32
SEED = 42

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Split data lokal (85% train, 15% validasi)

In [ ]:
def split_data(src, frac, seed):
    tr, va = Path("my_train"), Path("my_val")
    if va.exists():
        return tr, va
    rng = random.Random(seed)
    for cls in sorted(Path(src).iterdir()):
        if not cls.is_dir():
            continue
        files = sorted(cls.glob("*.jpg"))
        rng.shuffle(files)
        n_val = int(len(files) * frac)
        for i, f in enumerate(files):
            out = (va if i < n_val else tr) / cls.name
            out.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, out / f.name)
    return tr, va


tr_dir, va_dir = split_data(DATA, 0.15, SEED)

## Dataset dan model

In [ ]:
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    norm,
])

tr_ds = datasets.ImageFolder(tr_dir, train_tf)
va_ds = datasets.ImageFolder(va_dir, get_transform())
print(tr_ds.classes, len(tr_ds), len(va_ds))

tr_dl = DataLoader(tr_ds, BATCH, shuffle=True, num_workers=2)
va_dl = DataLoader(va_ds, BATCH, num_workers=2)

model = get_model()
pre = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).state_dict()
pre = {k: v for k, v in pre.items() if not k.startswith("fc.")}
model.load_state_dict(pre, strict=False)
model.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

## Training

In [ ]:
Path("submission").mkdir(exist_ok=True)
best = 0.0
hist = {"loss": [], "f1": []}

for ep in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        total += loss.item() * len(x)
    sched.step()

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in va_dl:
            y_pred += model(x.to(device)).argmax(1).cpu().tolist()
            y_true += y.tolist()
    m = metrics(y_true, y_pred)

    hist["loss"].append(total / len(tr_ds))
    hist["f1"].append(m["macro_f1"])
    print(f"epoch {ep} loss={hist['loss'][-1]:.4f} macro_f1={m['macro_f1']:.4f} acc={m['accuracy']:.4f}")

    if m["macro_f1"] > best:
        best = m["macro_f1"]
        torch.save(model.state_dict(), "submission/model.pth")

print("best macro_f1", round(best, 4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(hist["loss"]); ax[0].set_title("train loss")
ax[1].plot(hist["f1"]); ax[1].set_title("val macro-F1")
plt.show()

## Uji dengan skrip evaluasi resmi

In [ ]:
!python evaluate.py --submission ./submission --data ./my_val